## Carga de librerías

In [2]:
import pandas as pd
import numpy as np
from joblib import load

## Carga de datos

In [3]:
in_file = "data_clean/DALYs_clean_unificado_NEW.pkl"
with open(in_file, 'rb') as f:
    df = load(f)

print(f"Dimensiones: {df.shape}")
print(f"Columnas disponibles: {df.columns.tolist()}")

Dimensiones: (242956, 26)
Columnas disponibles: ['ano', 'pais', 'codigo_pais', 'codigo_ghe', 'categoria_principal', 'categoria_nivel1', 'categoria_nivel2', 'causa', 'sexo', 'edad', 'dalys', 'pib_per_capita', 'idh', 'indice_educacion', 'indice_salud', 'indice_ingresos', 'esperanza_vida', 'poblacion_miles', 'co2_anual', 'precipitacion_total', 'temperatura_superficial', 'humedad_relativa', 'poblacion_abs', 'tasa_dalys_100k', 'dalys_por_intervencion', 'desarrollo_vs_carga']


In [4]:
df.columns

Index(['ano', 'pais', 'codigo_pais', 'codigo_ghe', 'categoria_principal',
       'categoria_nivel1', 'categoria_nivel2', 'causa', 'sexo', 'edad',
       'dalys', 'pib_per_capita', 'idh', 'indice_educacion', 'indice_salud',
       'indice_ingresos', 'esperanza_vida', 'poblacion_miles', 'co2_anual',
       'precipitacion_total', 'temperatura_superficial', 'humedad_relativa',
       'poblacion_abs', 'tasa_dalys_100k', 'dalys_por_intervencion',
       'desarrollo_vs_carga'],
      dtype='object')

## Creación de tablas relevantes

###  Resumen global anual

#### ¿Cómo ha evolucionado la carga de NTDs (DALYs) a nivel global?


In [6]:
#================================================================
# ¿Cómo ha evolucionado la carga de NTDs (DALYs) a nivel global?
#================================================================

df_global_anual = df.groupby('ano', as_index=False)['dalys'].sum()

# Añadimos variables contextuales globales (promedio del IDH, PIB per cápita, etc.)
# Tomamos el primer valor por año, ya que son constantes por país y año en el dataset.
vars_contextuales = ['pib_per_capita', 'idh', 'indice_educacion', 'indice_salud',
                     'indice_ingresos', 'esperanza_vida', 'co2_anual',
                     'precipitacion_total', 'temperatura_superficial', 'humedad_relativa']

# Agrupamos por 'ano' y tomamos la media global de las variables contextuales.
df_contexto_global = df.groupby('ano', as_index=False)[vars_contextuales].mean()

# Unimos ambas tablas
df_global_anual = pd.merge(df_global_anual, df_contexto_global, on='ano')

print("     Descripción: Evolución anual de DALYs globales y promedios de variables contextuales.")
print(f"     Dimensiones: {df_global_anual.shape}")
display(df_global_anual)


     Descripción: Evolución anual de DALYs globales y promedios de variables contextuales.
     Dimensiones: (6, 12)


,ano,dalys,pib_per_capita,idh,indice_educacion,indice_salud,indice_ingresos,esperanza_vida,co2_anual,precipitacion_total,temperatura_superficial,humedad_relativa
0,2000,96394.592419,20084.238798,0.663007,0.560349,0.731982,0.682472,67.580016,106748.436508,0.037698,18.107159,67.909365
1,2010,128658.545509,22319.164497,0.697501,0.626672,0.771917,0.699789,70.176377,129422.023944,0.041746,18.636183,67.752606
2,2015,114224.924869,23658.876122,0.716944,0.655067,0.794535,0.712978,71.645699,142843.855843,0.035925,18.973142,66.227502
3,2019,103532.768150,25235.515657,0.725690,0.671764,0.809733,0.714960,72.633435,148600.469402,0.038382,19.166499,66.304798
4,2020,111317.225485,23039.495561,0.717215,0.670536,0.795663,0.701733,71.719211,132507.623537,0.040720,19.555624,66.871151
5,2021,107813.709352,24499.479328,0.716311,0.671603,0.787284,0.705840,71.175282,138168.500000,0.039406,19.376361,66.591235


### Resumen anual por país

In [9]:
#================================================================
# ¿Cómo ha evolucionado la carga de NTDs (DALYs) a nivel regional?
#================================================================

# Agrupamos por 'ano' y 'pais' (y añadimos 'codigo_pais' para facilitar mapas)
df_pais_anual = df.groupby(['ano', 'pais', 'codigo_pais']).agg({
    'dalys': 'sum',
    'tasa_dalys_100k': 'first', # La tasa es constante por país y año en este dataset
    'poblacion_abs': 'first'    # La población es constante por país y año
}).reset_index()

# Añadimos las variables contextuales que son constantes por país y año
df_contexto_pais = df[['ano', 'pais', 'codigo_pais'] + vars_contextuales].drop_duplicates()

# Unimos
df_pais_anual = pd.merge(df_pais_anual, df_contexto_pais, on=['ano', 'pais', 'codigo_pais'], how='inner')

print("     Descripción: DALYs totales por país y año, con variables contextuales específicas de cada país.")
print(f"     Dimensiones: {df_pais_anual.shape}")
display(df_pais_anual)


     Descripción: DALYs totales por país y año, con variables contextuales específicas de cada país.
     Dimensiones: (1738, 16)


C:\Users\troop\AppData\Local\Temp\ipykernel_24496\3245769102.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_pais_anual = df.groupby(['ano', 'pais', 'codigo_pais']).agg({


,ano,pais,codigo_pais,dalys,tasa_dalys_100k,poblacion_abs,pib_per_capita,idh,indice_educacion,indice_salud,indice_ingresos,esperanza_vida,co2_anual,precipitacion_total,temperatura_superficial,humedad_relativa
0,2000,Afghanistan,AFG,367.709300,1.700371,2152578.0,1617.8264,NaN,0.283,0.558,NaN,53.76,511.1,0.009,9.687,40.39
1,2000,Afghanistan,AFG,367.709300,1.700371,2152578.0,1617.8264,NaN,0.119,0.529,NaN,56.86,511.1,0.009,9.687,40.39
2,2000,Albania,ALB,2.260270,0.094449,144500.0,6262.7900,0.701,0.614,0.848,0.662,72.65,13719.0,0.032,12.860,68.98
3,2000,Albania,ALB,2.260270,0.094449,144500.0,6262.7900,0.649,0.564,0.860,0.563,78.42,13719.0,0.032,12.860,68.98
4,2000,Algeria,DZA,45.856595,0.103186,1569232.0,11558.2210,0.697,0.563,0.793,0.757,69.07,18827.0,0.007,18.430,46.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1733,2021,Yemen,YEM,514.715920,1.907674,3059985.0,NaN,0.220,0.250,0.687,0.062,67.13,7275.0,0.012,24.020,45.77
1734,2021,Zambia,ZMB,1465.961714,11.686986,1527155.0,3503.0350,0.583,0.583,0.631,0.541,58.49,2265.0,0.035,22.240,57.12
1735,2021,Zambia,ZMB,1465.961714,11.686986,1527155.0,3503.0350,0.548,0.522,0.637,0.494,63.93,2265.0,0.035,22.240,57.12
1736,2021,Zimbabwe,ZWE,241.249337,1.005033,1136548.0,4827.0890,0.560,0.636,0.596,0.463,56.23,26239.0,0.026,21.090,57.65


### Resumen por Enfermedad (Causa) a nivel global

In [8]:
#================================================================
# ¿Qué enfermedades contribuyen más a la carga?
#================================================================

df_causa_anual = df.groupby(['ano', 'causa'], as_index=False)['dalys'].sum()
df_causa_anual.rename(columns={'dalys': 'dalys_por_causa'}, inplace=True)

print("     Descripción: DALYs totales por año y tipo de NTD.")
print(f"     Dimensiones: {df_causa_anual.shape}")
display(df_causa_anual)

     Descripción: DALYs totales por año y tipo de NTD.
     Dimensiones: (108, 3)


C:\Users\troop\AppData\Local\Temp\ipykernel_24496\793016366.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_causa_anual = df.groupby(['ano', 'causa'], as_index=False)['dalys'].sum()


,ano,causa,dalys_por_causa
0,2000,Leprosy,34.440468
1,2000,a.Ascariasis,1521.085366
2,2000,a.Malaria,29758.887069
3,2000,b.Trichuriasis,285.320230
4,2000,b.Trypanosomiasis,164.770088
...,...,...,...
103,2021,i.Echinococcosis,508.986573
104,2021,j.Dengue,1723.052961
105,2021,k.Trachoma,125.079668
106,2021,l.Yellow fever,440.055187


### Resumen por país, año y enfermedad 
#### ¿Cómo se distribuyen las enfermedades por país?

In [11]:
#================================================================
# ¿Cómo se distribuyen las enferemdades por país?
#================================================================

df_pais_causa_anual = df.groupby(['ano', 'causa', 'codigo_pais', 'pais'], as_index=False)['dalys'].agg({
    'dalys': 'sum',
    'poblacion_abs': 'first' # La población es la misma para todas las filas de un mismo país/año
}).reset_index()

df_pais_causa_anual.rename(columns={'dalys': 'dalys_causa_pais'}, inplace=True)

# Añadimos variables contextuales (a nivel de país)
df_pais_causa_anual = pd.merge(df_pais_causa_anual, df_contexto_pais, on=['ano', 'pais', 'codigo_pais'])

print("     Descripción: DALYs por país, año y tipo de NTD, con contexto socioeconómico y ambiental.")
print(f"     Dimensiones: {df_pais_causa_anual.shape}")
display(df_pais_causa_anual)

C:\Users\troop\AppData\Local\Temp\ipykernel_24496\2999328010.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_pais_causa_anual = df.groupby(['ano', 'causa', 'codigo_pais', 'pais'], as_index=False)['dalys'].agg({
C:\Users\troop\AppData\Local\Temp\ipykernel_24496\2999328010.py:5: FutureWarning: Passing a dictionary to SeriesGroupBy.agg is deprecated and will raise in a future version of pandas. Pass a list of aggregations instead.
  df_pais_causa_anual = df.groupby(['ano', 'causa', 'codigo_pais', 'pais'], as_index=False)['dalys'].agg({


     Descripción: DALYs por país, año y tipo de NTD, con contexto socioeconómico y ambiental.
     Dimensiones: (31284, 17)


,index,ano,causa,codigo_pais,pais,dalys_causa_pais,poblacion_abs,pib_per_capita,idh,indice_educacion,indice_salud,indice_ingresos,esperanza_vida,co2_anual,precipitacion_total,temperatura_superficial,humedad_relativa
0,0,2000,Leprosy,AFG,Afghanistan,0.007397,0.000011,1617.8264,NaN,0.283,0.558,NaN,53.76,511.1,0.009,9.687,40.39
1,0,2000,Leprosy,AFG,Afghanistan,0.007397,0.000011,1617.8264,NaN,0.119,0.529,NaN,56.86,511.1,0.009,9.687,40.39
2,371,2000,Leprosy,ALB,Albania,0.000000,0.000000,6262.7900,0.701,0.614,0.848,0.662,72.65,13719.0,0.032,12.860,68.98
3,371,2000,Leprosy,ALB,Albania,0.000000,0.000000,6262.7900,0.649,0.564,0.860,0.563,78.42,13719.0,0.032,12.860,68.98
4,727,2000,Leprosy,ARE,United Arab Emirates,0.007057,0.000003,95390.7000,0.787,0.569,0.856,1.000,73.17,107078.0,0.000,27.860,43.23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31279,3695897,2021,m.Rabies,ZAF,South Africa,4.661130,0.341717,13681.9550,0.714,0.796,0.654,0.699,65.00,92211.0,0.024,17.700,52.82
31280,3696112,2021,m.Rabies,ZMB,Zambia,7.715420,1.033244,3503.0350,0.583,0.583,0.631,0.541,58.49,2265.0,0.035,22.240,57.12
31281,3696112,2021,m.Rabies,ZMB,Zambia,7.715420,1.033244,3503.0350,0.548,0.522,0.637,0.494,63.93,2265.0,0.035,22.240,57.12
31282,3696298,2021,m.Rabies,ZWE,Zimbabwe,5.716327,0.170872,4827.0890,0.560,0.636,0.596,0.463,56.23,26239.0,0.026,21.090,57.65
